# 01 · Data Collection & Processing
**Project**: War Shocks & Global Financial Risk Spillover  
**Period**: 2006-01-01 → 2025-12-31  
**Output**: `data/processed/` — all_variables_aligned, log_returns, rolling_volatility, war_dummies

All data collection and processing logic lives in `src/data_loader.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import (
    load_yahoo_equity, load_csi300, load_yahoo_controls, load_fred,
    merge_and_align, compute_returns, compute_volatility,
    build_war_dummies, save,
)

START = "2006-01-01"
END   = "2025-12-31"

## Step 1 · Load Equity Indices

In [2]:
print("=" * 55)
print("Yahoo Finance — Equity Indices")
print("=" * 55)
equity_frames = load_yahoo_equity(START, END)
csi300 = load_csi300(START, END)

Yahoo Finance — Equity Indices
  [OK] SP500      (^GSPC       ): 5030 rows | 2006-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] STOXX600   (^STOXX      ): 5019 rows | 2006-01-02 ~ 2025-12-30 | missing=0.0%
  [OK] FTSE100    (^FTSE       ): 5050 rows | 2006-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] DAX        (^GDAXI      ): 5074 rows | 2006-01-02 ~ 2025-12-30 | missing=0.0%
  [OK] Nikkei     (^N225       ): 4893 rows | 2006-01-04 ~ 2025-12-30 | missing=0.0%
  [OK] HangSeng   (^HSI        ): 4924 rows | 2006-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] MSCI_EM    (EEM         ): 5030 rows | 2006-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] CSI300     (local Excel ): 4860 rows | 2006-01-04 ~ 2025-12-31 | missing=0.0%


## Step 2 · Load Control Variables

In [3]:
print("=" * 55)
print("Yahoo Finance — Safe-haven & Control Variables")
print("=" * 55)
control_frames = load_yahoo_controls(START, END)

Yahoo Finance — Safe-haven & Control Variables
  [OK] Gold     (GC=F        ): 5029 rows
  [OK] DXY      (DX-Y.NYB    ): 5033 rows
  [OK] Silver   (SI=F        ): 5030 rows


## Step 3 · Load FRED Macro Variables

In [4]:
print("=" * 55)
print("FRED — Local Excel")
print("=" * 55)
fred_df = load_fred(START, END)

FRED — Local Excel
  [FIX] Brent: replaced 159 zero(s) → NaN
  [FIX] WTI: replaced 198 zero(s) → NaN
  [FIX] US10Y: replaced 214 zero(s) → NaN
  [FIX] HY_OAS: replaced 62 zero(s) → NaN
  [OK] FRED: 5217 rows | 2006-01-03 ~ 2025-12-31
    VIX       : 3.05% missing
    Brent     : 3.05% missing
    WTI       : 3.80% missing
    US10Y     : 4.10% missing
    HY_OAS    : 1.19% missing


## Step 4 · Merge & Align to S&P 500 Trading Days

In [5]:
print("=" * 55)
print("Merging & Aligning")
print("=" * 55)
combined = merge_and_align(equity_frames, control_frames, fred_df, csi300)
print(f"\nFinal shape : {combined.shape}")
print(f"Date range  : {combined.index[0].date()} ~ {combined.index[-1].date()}")
combined.head(10)

Merging & Aligning
  Trading days (S&P 500 basis): 5030

Final shape : (5030, 16)
Date range  : 2006-01-03 ~ 2025-12-30


,SP500,STOXX600,FTSE100,DAX,Nikkei,HangSeng,MSCI_EM,Gold,DXY,Silver,CSI300,VIX,Brent,WTI,US10Y,HY_OAS
Date,,,,,,,,,,,,,,,,
2006-01-03,1268.800049,313.040009,5681.500000,5460.680176,NaN,14944.769531,20.246731,530.700012,89.839996,9.087,NaN,11.14,61.51,63.11,4.37,3.73
2006-01-04,1273.459961,316.140015,5714.600098,5523.620117,16361.540039,15200.059570,20.430290,533.900024,89.139999,9.102,941.43,11.37,61.25,63.41,4.36,3.69
2006-01-05,1273.479980,315.040009,5691.200195,5516.529785,16425.369141,15271.129883,20.534229,526.299988,89.330002,8.809,959.13,11.31,61.68,62.81,4.36,3.64
2006-01-06,1285.449951,317.100006,5731.799805,5536.319824,16428.210938,15344.440430,20.963272,539.700012,88.849998,9.112,970.03,11.00,62.43,64.21,4.38,3.56
2006-01-09,1290.150024,317.709991,5731.500000,5537.109863,16428.210938,15547.429688,21.168945,549.099976,89.250000,9.222,975.25,11.13,62.51,63.56,4.38,3.51
2006-01-10,1289.689941,315.700012,5688.799805,5494.709961,16124.349609,15569.910156,20.914616,544.299988,89.330002,8.957,978.15,10.86,62.32,63.41,4.43,3.50
2006-01-11,1294.180054,317.549988,5731.500000,5532.890137,16363.589844,15650.879883,21.160099,548.799988,88.989998,9.007,973.48,10.94,61.54,63.91,4.46,3.47
2006-01-12,1286.060059,318.679993,5735.200195,5542.129883,16445.189453,15719.370117,20.865961,548.299988,89.440002,9.007,983.72,11.20,62.95,63.96,4.42,3.53
2006-01-13,1287.609985,316.869995,5711.000000,5483.089844,16454.949219,15787.969727,20.888079,556.099976,88.889999,9.112,978.81,11.23,61.58,63.86,4.36,3.62


## Step 5 · Compute Returns & Rolling Volatility

In [6]:
returns_df = compute_returns(combined)
vol_df     = compute_volatility(returns_df, window=21)

print(f"Returns    shape : {returns_df.shape}")
print(f"Volatility shape : {vol_df.shape}")
print("\nReturns (equity only) — descriptive stats:")
equity_ret_cols = [c for c in returns_df.columns if c.endswith("_ret") and "Gold" not in c and "DXY" not in c and "Silver" not in c and "Brent" not in c and "WTI" not in c]
returns_df[equity_ret_cols].describe().round(6)

Returns    shape : (5030, 16)
Volatility shape : (5030, 8)

Returns (equity only) — descriptive stats:


d:\Miniconda\envs\python3_12_D200\Lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


,SP500_ret,STOXX600_ret,FTSE100_ret,DAX_ret,Nikkei_ret,HangSeng_ret,MSCI_EM_ret,CSI300_ret
count,5029.000000,5029.000000,5029.000000,5029.000000,5024.000000,5029.000000,5029.000000,4897.000000
mean,0.000337,0.000127,0.000111,0.000298,0.000227,0.000109,0.000198,0.000287
std,0.012300,0.011552,0.011140,0.013255,0.014586,0.015109,0.017569,0.015884
min,-0.127652,-0.121915,-0.115117,-0.130549,-0.132341,-0.146954,-0.176334,-0.130116
25%,-0.004048,-0.004770,-0.004614,-0.005418,-0.006267,-0.006854,-0.007649,-0.006279
50%,0.000741,0.000414,0.000362,0.000691,0.000000,0.000000,0.000853,0.000000
75%,0.005784,0.005697,0.005469,0.006750,0.007356,0.007451,0.008345,0.007462
max,0.109572,0.094100,0.093842,0.107975,0.132346,0.134068,0.205141,0.089309


## Step 6 · Build War Dummy Variables

In [7]:
print("=" * 55)
print("War Event Dummies")
print("=" * 55)
war_dummy = build_war_dummies(combined.index, END)
war_dummy.value_counts().sort_index()

War Event Dummies
  mideast_war    days : 3760
  high_intensity days : 597
  gfc_crisis     days : 209
  covid_crisis   days : 168
  any_crisis     days : 377


mideast_war  high_intensity  gfc_crisis  covid_crisis  any_crisis
0            0               0           0             0             1075
                             1           0             1              195
1            0               0           0             0             2995
                                         1             1              168
             1               0           0             0              583
                             1           0             1               14
Name: count, dtype: int64

## Step 7 · Save to `data/processed/`

In [8]:
print("Saving processed files...")
save(combined,    "all_variables_aligned.xlsx")
save(returns_df,  "log_returns.xlsx")
save(vol_df,      "rolling_volatility.xlsx")
save(war_dummy,   "war_dummies.xlsx")
print("\nAll files saved to data/processed/")

Saving processed files...
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\all_variables_aligned.xlsx  (5030, 16)
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\log_returns.xlsx  (5030, 16)
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\rolling_volatility.xlsx  (5030, 8)
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\war_dummies.xlsx  (5030, 5)

All files saved to data/processed/


---
**Next** → `02_eda.ipynb` for exploratory analysis  